# 02 - Multitask Training (RUL Regression + Fault Classification)

In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import os
import sys
sys.path.append(os.path.abspath("../src"))
from multitask_model import MultiTaskLSTM
import os

In [7]:
# Load data
X = np.load('../outputs/X.npy')
y_reg = np.load('../outputs/y_reg.npy')
y_cls = np.load('../outputs/y_cls.npy')

# Convert to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_reg_tensor = torch.tensor(y_reg, dtype=torch.float32)
y_cls_tensor = torch.tensor(y_cls, dtype=torch.float32)

# Train/Val split
from sklearn.model_selection import train_test_split
X_train, X_val, yreg_train, yreg_val, ycls_train, ycls_val = train_test_split(
    X_tensor, y_reg_tensor, y_cls_tensor, test_size=0.2, random_state=42)

train_loader = DataLoader(TensorDataset(X_train, yreg_train, ycls_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, yreg_val, ycls_val), batch_size=64)

In [8]:
# Model + Loss + Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskLSTM(input_dim=X.shape[2], hidden_dim=128, dropout=0.2).to(device)

loss_reg = nn.SmoothL1Loss()
loss_cls = nn.BCELoss()
lambda_reg = 0.6  # 权重均衡
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
for epoch in range(20):
    model.train()
    total_loss = 0
    for xb, yb_reg, yb_cls in train_loader:
        xb, yb_reg, yb_cls = xb.to(device), yb_reg.to(device), yb_cls.to(device)
        optimizer.zero_grad()
        out_reg, out_cls = model(xb)
        loss_rul = loss_reg(out_reg, yb_reg)
        loss_fault = loss_cls(out_cls, yb_cls)
        loss = lambda_reg * loss_rul + (1 - lambda_reg) * loss_fault
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # 验证
    model.eval()
    with torch.no_grad():
        val_loss, val_rul, val_cls = 0, 0, 0
        for xb, yb_reg, yb_cls in val_loader:
            xb, yb_reg, yb_cls = xb.to(device), yb_reg.to(device), yb_cls.to(device)
            out_reg, out_cls = model(xb)
            loss_rul = loss_reg(out_reg, yb_reg)
            loss_fault = loss_cls(out_cls, yb_cls)
            loss = lambda_reg * loss_rul + (1 - lambda_reg) * loss_fault
            val_loss += loss.item()
            val_rul += loss_rul.item()
            val_cls += loss_fault.item()

    print(f"Epoch {epoch+1:02d} | Train Loss: {total_loss/len(train_loader):.2f} | "
          f"Val Total: {val_loss/len(val_loader):.2f} | RUL Loss: {val_rul/len(val_loader):.2f} | Fault Loss: {val_cls/len(val_loader):.2f}")

Epoch 01 | Train Loss: 5.21 | Val Total: 5.59 | RUL Loss: 8.90 | Fault Loss: 0.63
Epoch 02 | Train Loss: 5.04 | Val Total: 5.40 | RUL Loss: 8.63 | Fault Loss: 0.56
Epoch 03 | Train Loss: 5.17 | Val Total: 5.17 | RUL Loss: 8.29 | Fault Loss: 0.48
Epoch 04 | Train Loss: 5.01 | Val Total: 4.90 | RUL Loss: 7.91 | Fault Loss: 0.39
Epoch 05 | Train Loss: 4.53 | Val Total: 4.67 | RUL Loss: 7.56 | Fault Loss: 0.33
Epoch 06 | Train Loss: 4.67 | Val Total: 4.44 | RUL Loss: 7.22 | Fault Loss: 0.28
Epoch 07 | Train Loss: 4.69 | Val Total: 4.24 | RUL Loss: 6.91 | Fault Loss: 0.24
Epoch 08 | Train Loss: 4.18 | Val Total: 4.06 | RUL Loss: 6.62 | Fault Loss: 0.22
Epoch 09 | Train Loss: 4.19 | Val Total: 3.90 | RUL Loss: 6.36 | Fault Loss: 0.21
Epoch 10 | Train Loss: 3.50 | Val Total: 3.76 | RUL Loss: 6.14 | Fault Loss: 0.20
Epoch 11 | Train Loss: 3.50 | Val Total: 3.64 | RUL Loss: 5.94 | Fault Loss: 0.20
Epoch 12 | Train Loss: 3.42 | Val Total: 3.54 | RUL Loss: 5.77 | Fault Loss: 0.20
Epoch 13 | Train

In [10]:
# Save model
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), '../models/multitask_lstm.pt')